In [19]:
# ── Dependency Guard ───────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.3.3


In [20]:
# ── Reproducibility Header ────────────────────────────────────────────

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [21]:
# ── Data Loading: F1 Results via Ergast API ──────────────────────────────────

import requests

def get_season_results(year: int) -> pd.DataFrame:
    """Fetch all race results for a given season via Ergast API."""
    offset = 0
    limit = 100
    rows = []

    while True:
        url = f"https://api.jolpi.ca/ergast/f1/{year}/results.json?limit={limit}&offset={offset}"
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json()['MRData']
        total = int(data['total'])
        races = data['RaceTable']['Races']

        for race in races:
            for result in race['Results']:
                rows.append({
                    'season': int(race['season']),
                    'round': int(race['round']),
                    'race_name': race['raceName'],
                    'circuit': race['Circuit']['circuitId'],
                    'date': race['date'],
                    'driver': result['Driver']['driverId'],
                    'driver_name': f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
                    'constructor': result['Constructor']['constructorId'],
                    'grid': int(result['grid']),
                    'position': int(result['position']) if result['position'].isdigit() else None,
                    'points': float(result['points']),
                    'status': result['status'],
                    'laps': int(result['laps']),
                })

        offset += limit
        if offset >= total:
            break

    return pd.DataFrame(rows)

# Load data for 2022, 2023, 2024
print("Loading F1 race data...")
seasons = [2022, 2023, 2024]
dfs = []

for year in seasons:
    df_year = get_season_results(year)
    print(f"  {year}: {len(df_year)} results from {df_year['round'].max()} races")
    dfs.append(df_year)

df_all = pd.concat(dfs, ignore_index=True)
df_all['date'] = pd.to_datetime(df_all['date'])


df_all['top10'] = (df_all['position'] <= 10) 



Loading F1 race data...
  2022: 440 results from 22 races
  2023: 440 results from 22 races
  2024: 479 results from 24 races


The acuarcy of the method was 67.23% for the lab 2 we need to at least get that accurate.

In [22]:
# ── 4.1 Domain Heuristic Baseline ────────────────────────────────────────────
# Rule: "If grid ≤ 10 predict top-10 finish; otherwise predict non-top-10"
# Rationale: Starting position is strongly correlated with finishing in top-10.
#            Drivers starting from pitlane (grid=0) are treated conservatively.

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns


print(f"Records with finished status: {len(df_all)}")
print(f"Top-10 ratio: {df_all['top10'].mean():.2%}")

# ── Train/Validation Split ──────────────────────────────────────────────────
# Split by race to ensure no race appears in both sets
races_list = (
    df_all[["season", "round"]]
    .drop_duplicates()
    .sort_values(["season", "round"])
)

train_races, val_races = train_test_split(
    races_list, test_size=0.3, random_state=RANDOM_SEED
)
# More robust split approach
def get_race_ids(seasons, rounds):
    """Create a set of race IDs for splitting."""
    return set(zip(seasons, rounds))

train_race_ids = get_race_ids(train_races['season'].values, train_races['round'].values)
val_race_ids = get_race_ids(val_races['season'].values, val_races['round'].values)

# Apply split
df_all['race_id'] = list(zip(df_all['season'], df_all['round']))
df_all['dataset'] = df_all['race_id'].apply(
    lambda x: 'train' if x in train_race_ids else 'val'
)

df_train = df_all[df_all['dataset'] == 'train'].copy()
df_val = df_all[df_all['dataset'] == 'val'].copy()

print(f"\nTrain set: {len(df_train)} records from {df_train['race_id'].nunique()} races")
print(f"Validation set: {len(df_val)} records from {df_val['race_id'].nunique()} races")
print(f"Train top-10 ratio: {df_train['top10'].mean():.2%}")
print(f"Validation top-10 ratio: {df_val['top10'].mean():.2%}")

# ── 4.1 Apply Baseline Rule ──────────────────────────────────────────────────
# Baseline rule: grid_position ≤ 10 → predict top-10; otherwise → predict non-top-10

def baseline_prediction(grid_pos):
    """
    Rule-based heuristic: If driver started in top 10, predict they finish in top 10.
    """
    if pd.isna(grid_pos):
        return False
    return grid_pos <= 10 and grid_pos > 0  # Exclude grid=0 (pit lane starts)

# Apply predictions on validation set
df_val['predicted_top10'] = df_val['grid'].apply(baseline_prediction)
df_val['actual_top10'] = df_val['top10'].astype(bool)

print("\n" + "="*60)
print("BASELINE PREDICTIONS (Validation Set)")
print("="*60)
print(f"Predictions made: {len(df_val)}")
print(f"Predicted top-10: {df_val['predicted_top10'].sum()}")
print(f"Actual top-10: {df_val['actual_top10'].sum()}")


Records with finished status: 1359
Top-10 ratio: 50.04%

Train set: 939 records from 47 races
Validation set: 420 records from 21 races
Train top-10 ratio: 50.05%
Validation top-10 ratio: 50.00%

BASELINE PREDICTIONS (Validation Set)
Predictions made: 420
Predicted top-10: 210
Actual top-10: 210


In [25]:
# ── 4.2 Calculate Metrics ───────────────────────────────────────────────────

# Extract predictions and actuals
y_true = df_val['actual_top10'].values
y_pred = df_val['predicted_top10'].values

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\n" + "="*60)
print("METRICS - VALIDATION SET")
print("="*60)
print(f"✓ Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"✓ Recall    : {recall:.4f} (True Positive Rate)")
print(f"✓ Precision : {precision:.4f} (Positive Predictive Value)")
print(f"✓ F1-Score  : {f1:.4f} (Harmonic mean)")

print(f"\n{'─'*60}")
print("Confusion Matrix:")
print(f"{'─'*60}")
print(f"TN (Correct Non-Top10): {cm[0, 0]}")
print(f"FP (Predicted Top10, Actually Not): {cm[0, 1]}")
print(f"FN (Predicted Not Top10, Actually Top10): {cm[1, 0]}")
print(f"TP (Correct Top10): {cm[1, 1]}")

print(f"\n{'─'*60}")
print("Classification Report:")
print(f"{'─'*60}")
print(classification_report(y_true, y_pred, target_names=['Non-Top10', 'Top10']))



METRICS - VALIDATION SET
✓ Accuracy  : 0.7476 (74.76%)
✓ Recall    : 0.7476 (True Positive Rate)
✓ Precision : 0.7476 (Positive Predictive Value)
✓ F1-Score  : 0.7476 (Harmonic mean)

────────────────────────────────────────────────────────────
Confusion Matrix:
────────────────────────────────────────────────────────────
TN (Correct Non-Top10): 157
FP (Predicted Top10, Actually Not): 53
FN (Predicted Not Top10, Actually Top10): 53
TP (Correct Top10): 157

────────────────────────────────────────────────────────────
Classification Report:
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

   Non-Top10       0.75      0.75      0.75       210
       Top10       0.75      0.75      0.75       210

    accuracy                           0.75       420
   macro avg       0.75      0.75      0.75       420
weighted avg       0.75      0.75      0.75       420



## 4.3 Reflection on Baseline Accuracy

### Is this accuracy good enough to make decisions with?

The baseline achieves an accuracy of approximately **{accuracy*100:.1f}%** on the validation set using a simple rule: *"If grid position ≤ 10, predict top-10 finish."*

**Can we make decisions with this?** It depends on context:
- For **rough screening** (e.g., identifying unlikely top-10 candidates), this accuracy is reasonable
- For **critical decisions** (betting, season strategies), we'd want >80%+ accuracy minimum
- The **recall score ({recall:.1f}%)** matters more than accuracy if we care about catching ALL top-10 finishers
- The **precision score ({precision:.1f}%)** matters more if we want to avoid false alarms

### What could accuracy be hiding?

Accuracy is a **dangerous metric** in imbalanced datasets. Consider this scenario:

**If 50% of drivers finish in top-10:**
- A naive baseline that **always predicts "top-10"** would achieve **50% accuracy** (half are correct by chance)
- But it would have **100% recall** (catches all top-10 finishers) and **50% precision** (many false positives)
- Another naive baseline that **always predicts "non-top-10"** would also achieve **50% accuracy**

**In our actual data:**
- Top-10 ratio: **{(df_val['actual_top10'].mean()*100):.1f}%** (see above)
- Our baseline accuracy: **{accuracy*100:.1f}%**

**What accuracy hides:**
1. **Class imbalance**: Accuracy doesn't reveal if we're biased toward the majority class
2. **False negatives vs. False positives**: Missing a top-10 finish might be worse than a false alarm
3. **Feature leakage**: Grid position is **pre-race info** (no leakage ✓), but a more complex model might leak race information
4. **Generalization**: Good validation accuracy doesn't guarantee performance on future races

**Better decision framework:**
- Use **F1-score** for balanced evaluation ({f1:.1f}%)
- Use **confusion matrix** to understand error types
- Use **business metrics** (e.g., "What % of our predictions are actionable?")
- Compare against **multiple baselines** (always top-10, always non-top-10, random)
- Evaluate on **unseen tracks/seasons** to test generalization

BASELINE RULE: **Grid Position ≤ 10 → Top-10 Finish**
